Web Scraping

In this bonus section, a simple web scraper was implemented to collect data for 50 Samand cars manufactured after 1385 from bama.ir.

The extracted fields include price, mileage, color, production year, transmission type, and description. The collected data is then saved as an Excel file.

In [21]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [22]:
options = Options()                         #creating browser
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)                      


Number processing functions

In [23]:
def normalize_digits(text):
    if text is None:
        return None
    
    text = str(text)
    
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    arabic_digits = "٠١٢٣٤٥٦٧٨٩"
    english_digits = "0123456789"
    
    for p, e in zip(persian_digits, english_digits):
        text = text.replace(p, e)
    
    for a, e in zip(arabic_digits, english_digits):
        text = text.replace(a, e)
    
    return text


def extract_number(text):
    if text is None or pd.isna(text):
        return pd.NA
    
    text = normalize_digits(text)
    match = re.search(r"\d[\d,]*", text)
    
    if match:
        return int(match.group().replace(",", ""))
    
    return pd.NA

Label values

In [24]:
def find_value_after_label(lines, labels):
    for label in labels:
        for i, line in enumerate(lines):
            line = line.strip()
            
            if line == label:
                for j in range(i + 1, min(i + 4, len(lines))):
                    candidate = lines[j].strip()
                    
                    if candidate and candidate not in labels:
                        return candidate
            
            if line.startswith(label + " "):
                value = line.replace(label, "", 1).strip(" :")
                if value:
                    return value
            
            if line.startswith(label + ":"):
                value = line.replace(label + ":", "", 1).strip()
                if value:
                    return value
    
    return pd.NA

Description

In [25]:
def extract_description(lines):
    stop_words = [
        "مشخصات",
        "قیمت",
        "نمایش شماره",
        "گزارش",
        "آگهی‌های مشابه",
        "امکانات",
        "تماس"
    ]
    
    for i, line in enumerate(lines):
        if line.strip() == "توضیحات":
            desc_lines = []
            
            for next_line in lines[i + 1:]:
                next_line = next_line.strip()
                
                if next_line in stop_words:
                    break
                
                if next_line:
                    desc_lines.append(next_line)
            
            description = " ".join(desc_lines).strip()
            return description if description else pd.NA
    
    return pd.NA

Ad data was parsed

In [26]:
def parse_ad_text(text, url):
    text = normalize_digits(text)
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    full_text = " ".join(lines)
    
    title = pd.NA
    for line in lines:
        if "سمند" in line:
            title = line
            break
    
    year_matches = re.findall(r"\b(13[8-9]\d|14[0-5]\d)\b", full_text)
    production_year = int(year_matches[0]) if year_matches else pd.NA
    
    if "صفر کیلومتر" in full_text:
        mileage = 0
    else:
        mileage_text = find_value_after_label(lines, ["کارکرد", "کارکرد (کیلومتر)"])
        mileage = extract_number(mileage_text)
        
        if pd.isna(mileage):
            mileage_match = re.search(r"(\d[\d,]*)\s*کیلومتر", full_text)
            mileage = extract_number(mileage_match.group()) if mileage_match else pd.NA
    
    if "قیمت توافقی" in full_text or "توافقی" in full_text:
        price = pd.NA
    else:
        price_matches = re.findall(r"\d[\d,]*\s*تومان", full_text)
        price = extract_number(price_matches[0]) if price_matches else pd.NA
    
    body_color = find_value_after_label(lines, ["رنگ بدنه"])
    
    if pd.isna(body_color):
        color_match = re.search(r"رنگ بدنه\s+([^\s]+)", full_text)
        body_color = color_match.group(1) if color_match else pd.NA
    
    transmission_text = find_value_after_label(lines, ["گیربکس"])
    
    if not pd.isna(transmission_text):
        if "اتومات" in transmission_text:
            transmission = "automatic"
        elif "دنده" in transmission_text:
            transmission = "manual"
        else:
            transmission = transmission_text
    else:
        if "اتومات" in full_text:
            transmission = "automatic"
        elif "دنده ای" in full_text or "دنده‌ای" in full_text:
            transmission = "manual"
        else:
            transmission = pd.NA
    
    description = extract_description(lines)
    
    return {
        "title": title,
        "price": price,
        "mileage": mileage,
        "color": body_color,
        "production_year": production_year,
        "transmission_type": transmission,
        "description": description,
        "url": url
    }

Ad links were collected

In [27]:
def collect_ad_links(driver, start_url, target_links=250, max_scrolls=100):
    driver.get(start_url)
    time.sleep(5)
    
    ad_links = set()
    no_new_counter = 0
    
    last_count = 0
    
    for _ in range(max_scrolls):
        links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/car/detail-"]')
        
        for link in links:
            href = link.get_attribute("href")
            
            if href:
                clean_href = href.split("?")[0]
                ad_links.add(clean_href)
        
        if len(ad_links) == last_count:
            no_new_counter += 1
        else:
            no_new_counter = 0
        
        last_count = len(ad_links)
        
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        if len(ad_links) >= target_links:
            break
        
        if no_new_counter >= 12:
            break
    
    return list(ad_links)

There was a problem, as the main link returned only 30 ads, so I split prices into ranges and used 5 different links.(300to500,500to700,700to1000,1000to1500,1500to3000).

In [28]:
search_urls = [
    "https://bama.ir/car/samand?installment=0&price=300000000,500000000",
    "https://bama.ir/car/samand?installment=0&price=500000000,700000000",
    "https://bama.ir/car/samand?installment=0&price=700000000,1000000000",
    "https://bama.ir/car/samand?installment=0&price=1000000000,1500000000",
    "https://bama.ir/car/samand?installment=0&price=1500000000,3000000000"
]

In [29]:
ad_links = set()

for search_url in search_urls:
    driver.get(search_url)
    time.sleep(5)
    
    for _ in range(20):
        links = driver.find_elements(By.CSS_SELECTOR, 'a[href*="/car/detail-"]')
        
        for link in links:
            href = link.get_attribute("href")
            if href:
                ad_links.add(href.split("?")[0])
        
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
    
    print(search_url)
    print("Total links so far:", len(ad_links))

print("Final number of collected links:", len(ad_links))

https://bama.ir/car/samand?installment=0&price=300000000,500000000
Total links so far: 22
https://bama.ir/car/samand?installment=0&price=500000000,700000000
Total links so far: 63
https://bama.ir/car/samand?installment=0&price=700000000,1000000000
Total links so far: 147
https://bama.ir/car/samand?installment=0&price=1000000000,1500000000
Total links so far: 235
https://bama.ir/car/samand?installment=0&price=1500000000,3000000000
Total links so far: 272
Final number of collected links: 272


Filtered valid ads.

In [32]:
records = []
checked_count = 0

for url in list(ad_links):
    driver.get(url)
    time.sleep(2.5)
    
    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        page_text = driver.find_element(By.TAG_NAME, "body").text
        
        record = parse_ad_text(page_text, url)
        checked_count += 1
        
        if not pd.isna(record["production_year"]):
            if record["production_year"] > 1385:
                records.append(record)
        
        print(f"Checked: {checked_count} | Valid records: {len(records)}")
        
        if len(records) >= 50:
            break
            
    except Exception as e:
        print("Error on:", url)
        print(e)

print("Final number of valid records:", len(records))

Checked: 1 | Valid records: 1
Checked: 2 | Valid records: 2
Checked: 3 | Valid records: 3
Checked: 4 | Valid records: 4
Checked: 5 | Valid records: 5
Checked: 6 | Valid records: 6
Checked: 7 | Valid records: 7
Checked: 8 | Valid records: 8
Checked: 9 | Valid records: 9
Checked: 10 | Valid records: 10
Checked: 11 | Valid records: 11
Checked: 12 | Valid records: 12
Checked: 13 | Valid records: 13
Checked: 14 | Valid records: 14
Checked: 15 | Valid records: 15
Checked: 16 | Valid records: 16
Checked: 17 | Valid records: 17
Checked: 18 | Valid records: 18
Checked: 19 | Valid records: 18
Checked: 20 | Valid records: 19
Checked: 21 | Valid records: 20
Checked: 22 | Valid records: 21
Checked: 23 | Valid records: 22
Checked: 24 | Valid records: 23
Checked: 25 | Valid records: 24
Checked: 26 | Valid records: 25
Checked: 27 | Valid records: 26
Checked: 28 | Valid records: 27
Checked: 29 | Valid records: 28
Checked: 30 | Valid records: 28
Checked: 31 | Valid records: 29
Checked: 32 | Valid record

DataFrame was created.

In [33]:
df_samand = pd.DataFrame(records)

required_columns = [
    "price",
    "mileage",
    "color",
    "production_year",
    "transmission_type",
    "description",
    "title",
    "url"
]

df_samand = df_samand[required_columns]

df_samand.head()

,price,mileage,color,production_year,transmission_type,description,title,url
0,1100000000,50320,سفید,1394,manual,بسیار تمیز و کم‌کارکرد. کیلومتر (واقعی) بدنه س...,سمند,https://bama.ir/car/detail-balxoqki-samand-lx-...
1,680000000,350000,سفید,1390,manual,سمند معمولی مدل 90 موتور تازه تعمیر شده لاستیک...,سمند,https://bama.ir/car/detail-jzt9gybv-samand-se-...
2,850000000,0,سفید,1387,manual,وضعیت فنی:سالم وضعیت بدنه:خیلی تمیز بدون هیچ ل...,سمند,https://bama.ir/car/detail-1086qasm-samand-sor...
3,1370000000,0,سفید,1405,manual,کارت طلایی مدارک آماده انتقال معاوضه با 207 صف...,سمند,https://bama.ir/car/detail-bfrfs2l6-samand-sor...
4,875000000,170000,سفید,1397,manual,باسلام... ماشین فوق العاده سالم کارکرد واقعی و...,سمند,https://bama.ir/car/detail-pymljzm0-samand-lx-...


In [34]:
df_samand.shape

(50, 8)

In [35]:
df_samand.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   price              39 non-null     object
 1   mileage            50 non-null     int64 
 2   color              50 non-null     str   
 3   production_year    50 non-null     int64 
 4   transmission_type  50 non-null     str   
 5   description        46 non-null     str   
 6   title              50 non-null     str   
 7   url                50 non-null     str   
dtypes: int64(2), object(1), str(5)
memory usage: 3.3+ KB


In [36]:
df_samand.describe()

,mileage,production_year
count,50.000000,50.000000
mean,139166.400000,1397.000000
std,139340.264626,5.931583
min,0.000000,1386.000000
25%,0.000000,1393.000000
50%,135000.000000,1396.500000
75%,265750.000000,1404.000000
max,400000.000000,1405.000000


In [37]:
df_samand.to_excel("samand_bama_bonus.xlsx", index=False, na_rep="NA")

In [38]:
driver.quit()